# Spatial Analysis: Land-Use Change (LUC) Emissions

**Objective:**
This script estimates the greenhouse gas (GHG) emissions associated with land-use change in 2022 coca cultivation sites.

**Methodology:**
1.  **Preprocessing:** The source emissions raster (TIF) is reprojected to the study area.
2.  **Zonal Statistics:** We calculate the **Total Cumulative Emissions (Sum)** for each 1km² coca grid cell using Zonal Statistics.
3.  **Normalization & Annualization:**
    * We attribute emissions to coca based on the proportion of the cell occupied by the crop.
    * We convert cumulative emissions (24-year model) to an annual rate.

**Formula for Annual LUC Emissions:**

$$Annual\ Rate\ (kt\ CO_{2}eq\ yr^{-1}) = \frac{\sum Emissions_{Total} \times \left( \frac{Area_{Coca}}{100} \right)}{24}$$

*Where:*
* **$\sum Emissions_{Total}$**: Sum of Mg CO2e within the 1km² grid cell.
* **$Area_{Coca} / 100$**: Hectares of coca normalized to the percentage of the 100ha (1km²) cell.
* **$24$**: The model period (2001–2024) to derive the annual average.

**Input Data:**
* **LUC Emissions (Raster):** ('10N_080W_gross_emis_all_gases_all_drivers_Mg_CO2e_pixel_biomass_soil_full_extent_2001_24') representing cumulative GHG emissions(Source: https://www.globalforestwatch.org/) (Harris et al., 2021).
* **Coca Grid (Vector):** 1km² grid cells with coca density (Source: https://www.biesimci.org/) (UNODC/BIESIMCI 2023).

## 1. Data Loading & Pre-processing Note

**Computational Constraint:**
The original source dataset (Harris et al., 2021) is a high-resolution (30m) raster. Reprojecting this dataset from WGS 1984 to the local UTM Zone 18N system is a computationally intensive process (taking >4 hours via standard geoprocessing tools) due to the pixel density and extent.

**Workflow Adjustment:**
To ensure this script remains efficient and reproducible, the reprojection was performed as an offline pre-processing step. We import the resulting **reprojected raster** directly.

**Input Data:**
* **LUC Emissions (Reprojected):** `10N_080W_gross_emis_all_gases_all_drivers_Mg_CO2e_pixel_biomass_soil_full_extent_2001_24_reprojected.tif` (Pre-processed to UTM Zone 18N).
* **Coca Grid:** `Grillas1k.shp` (1km² vector grid).

In [5]:
import arcpy
from arcpy.sa import *
import os

# =============================================================================
# 🟢 USER CONFIGURATION
# =============================================================================
workspace = r"M:\Documents\Spatial_Analysis_Project\data\interim\Spatial_Analysis\Spatial_Analysis.gdb"

# PATH TO THE PRE-PROCESSED (ALREADY REPROJECTED) RASTER
# Ideally, this file is located in your 'interim' or 'processed' data folder
input_reprojected_raster = r"M:\Documents\Spatial_Analysis_Project\data\interim\10N_080W_gross_emis_all_gases_all_drivers_Mg_CO2e_pixel_biomass_soil_full_extent_2001_24_reprojected.tif"

# Path to Shapefile
coca_shapefile = r"M:\Documents\Spatial_Analysis_Project\data\raw\Grillas1k_Coca_2022\Grillas1k.shp"
output_excel = r"M:\Documents\Spatial_Analysis_Project\data\interim\LUC_annual_rate_per_grid.xlsx"

# Fields
zone_id_field = "GRI_1K_F_1"
coca_area_field = "Coca_2022"

# =============================================================================
# VERIFICATION
# =============================================================================
arcpy.env.workspace = workspace
arcpy.env.overwriteOutput = True
arcpy.CheckOutExtension("Spatial")

print("--- Verifying Spatial Alignment ---")

# 1. Check if the pre-processed raster exists
if not os.path.exists(input_reprojected_raster):
    raise FileNotFoundError(f"❌ The pre-reprojected file was not found: {input_reprojected_raster}")

# 2. Check Coordinate Systems
desc_rast = arcpy.Describe(input_reprojected_raster)
desc_vect = arcpy.Describe(coca_shapefile)

print(f"Raster Projection: {desc_rast.spatialReference.name}")
print(f"Vector Projection: {desc_vect.spatialReference.name}")

# Warn if they don't match (though Zonal Stats can handle minor mismatches, it's best they match)
if desc_rast.spatialReference.name != desc_vect.spatialReference.name:
    print("⚠️ WARNING: Coordinate systems do not match perfectly. Check your pre-processing.")
else:
    print("✅ Coordinate systems align.")

--- Verifying Spatial Alignment ---
Raster Projection: WGS_1984_UTM_Zone_18N
Vector Projection: WGS_1984_UTM_Zone_18N
✅ Coordinate systems align.


## 3. Zonal Statistics & Annualization
With the data spatially aligned, we calculate the total emissions for each grid cell and attribute the share corresponding to coca cultivation.

**Methodology:**
1.  **Zonal Statistics (SUM):** We sum the values of all 30m pixels falling within each 1km² grid cell. Since the input pixel values represent absolute mass (Mg CO₂-eq), summing them provides the total emissions for the area.
2.  **Allocation:** We calculate the annual rate using the formula:
    $$Rate_{Annual} = \frac{\sum(Emissions_{Total}) \times \frac{Area_{Coca}}{100}}{24}$$

In [6]:
# Intermediate Outputs
zonal_stats_table = f"{workspace}\\LUC_Stats_Table"
final_layer_name = "LUC_Final_Layer"

try:
    print("Step 1: Calculating Zonal Statistics (SUM)...")
    print("   (This may take time depending on disk speed)...")
    
    # Calculate SUM of emissions per grid cell
    ZonalStatisticsAsTable(
        in_zone_data=coca_shapefile,
        zone_field=zone_id_field,
        in_value_raster=input_reprojected_raster,
        out_table=zonal_stats_table,
        ignore_nodata="DATA",
        statistics_type="SUM"
    )
    print("   ✅ Zonal Statistics complete.")

    print("Step 2: Linking results & Calculating...")
    
    # Create a working copy of the shapefile to avoid locking the raw data
    arcpy.conversion.FeatureClassToFeatureClass(coca_shapefile, workspace, final_layer_name)
    processing_layer = f"{workspace}\\{final_layer_name}"
    
    # Join the stats table (SUM field) to the shapefile
    arcpy.management.JoinField(processing_layer, zone_id_field, zonal_stats_table, zone_id_field, ["SUM"])

    # Add Field for Annual Emissions
    arcpy.management.AddField(processing_layer, "Ann_LUC_Emissions", "DOUBLE")
    
    # Calculate Formula
    # Logic: If SUM exists, use it; otherwise 0.
    expression = f"(!SUM! if !SUM! else 0) * (!{coca_area_field}! / 100) / 24"
    
    arcpy.management.CalculateField(
        in_table=processing_layer,
        field="Ann_LUC_Emissions",
        expression=expression,
        expression_type="PYTHON3"
    )

    print(f"Step 3: Exporting to {output_excel}...")
    arcpy.conversion.TableToExcel(processing_layer, output_excel)
    print("✅ SUCCESS: Analysis finished.")

except Exception as e:
    print(f"❌ Error during analysis: {e}")

Step 1: Calculating Zonal Statistics (SUM)...
   (This may take time depending on disk speed)...
   ✅ Zonal Statistics complete.
Step 2: Linking results & Calculating...
Step 3: Exporting to M:\Documents\Spatial_Analysis_Project\data\interim\LUC_annual_rate_per_grid.xlsx...
✅ SUCCESS: Analysis finished.
